Title

In [1]:
import requests
from langchain.tools import tool

@tool
def get_real_weather(city: str) -> str:
    """Get the REAL current weather for a city, using a live weather API."""
    # Step 1: geocode the city name into latitude/longitude
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    )
    geo_data = geo_response.json()
    if not geo_data.get("results"):
        return f"Could not find a location matching '{city}'."

    location = geo_data["results"][0]
    lat, lon = location["latitude"], location["longitude"]

    # Step 2: fetch the real current weather at that exact location
    weather_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": "true"},
    )
    weather_data = weather_response.json()
    current = weather_data.get("current_weather", {})

    return (
        f"Real current weather in {location['name']}, {location.get('country', '')}: "
        f"{current.get('temperature')}°C, wind {current.get('windspeed')} km/h."
    )


# Test the Weather tool
print(get_real_weather.invoke({"city":"Bengaluru"}))

Real current weather in Bengaluru, India: 21.2°C, wind 7.9 km/h.


In [2]:
import os 
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

load_dotenv()
tavily_api_key = os.environ["TAVILY_API_KEY"]
tavily_search = TavilySearch(max_results=5, topic="general")

# Test the web search tool
result = tavily_search.invoke({"query": "latest AI news"})
for idx, item in enumerate(result["results"]):
    print(idx+1, ": ", item["title"])

1 :  AI News | Latest News | Insights Powering AI-Driven Business Growth
2 :  Artificial Intelligence - Latest AI News and Analysis - WSJ.com
3 :  Latest News on AI - Artificial Intelligence (Generative) Resources - Guides at Georgetown University
4 :  AI News: OpenAI Unleashes TONS of new stuff!
5 :  AI News & Artificial Intelligence | TechCrunch


In [ ]:
import sqlite3
from pydantic import BaseModel

def setup_database():
  conn = sqlite3.connect("tripmate.db")
  cur = conn.cursor()
  cur.execute("""
    CREATE TABLE IF NOT EXISTS trips (
        trip_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT,
        destination TEXT,
        start_date TEXT,
        end_date TEXT,
        status TEXT DEFAULT 'confirmed'
    )
  """)
  conn.commit()
  conn.close()


@tool
def save_trip(user_id: str, destination: str, start_date: str, end_date: str) -> str:
    """Save a new trip to the real database."""
    conn = sqlite3.connect("tripmate.db")
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO trips (user_id, destination, start_date, end_date) VALUES (?, ?, ?, ?)",
        (user_id, destination, start_date, end_date),
    )
    conn.commit()
    trip_id = cur.lastrowid
    conn.close()
    return f"Trip #{trip_id} saved: {destination}, {start_date} to {end_date}."


@tool
def get_saved_trips(user_id: str) -> str:
    """Look up all saved trips for a user from the real database."""
    conn = sqlite3.connect("tripmate.db")
    cur = conn.cursor()
    cur.execute("SELECT trip_id, destination, start_date, end_date, status FROM trips WHERE user_id = ?", (user_id,))
    rows = cur.fetchall()
    conn.close()
    if not rows:
        return "No saved trips found for this user."
    return "\n".join(f"Trip #{r[0]}: {r[1]} ({r[2]} to {r[3]}) -- {r[4]}" for r in rows)


# Test Sqlite3 DB tool
setup_database()
print(save_trip.invoke({"user_id": "rohan_01", "destination": "Bali", "start_date": "2026-09-01", "end_date": "2026-09-10"}))
print(get_saved_trips.invoke({"user_id": "rohan_01"}))

Trip #2 saved: Bali, 2026-09-01 to 2026-09-10.
Trip #1: Bali (2026-09-01 to 2026-09-10) -- confirmed
Trip #2: Bali (2026-09-01 to 2026-09-10) -- confirmed


In [4]:
@tool
def book_premium_concierge(destination: str) -> str:
    """Book a dedicated human concierge for trip planning. Premium members only."""
    return f"Premium concierge assigned for your {destination} trip."

# Test Book Premium Concierge Tool
print(book_premium_concierge.invoke({"destination": "Bengaluru"}))

Premium concierge assigned for your Bengaluru trip.


In [5]:
# Dynamic Tool Gating
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

@wrap_model_call
def gate_premium_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only expose book_premium_concierge to premium members."""
    is_premium = request.runtime.context.is_premium_member
    if not is_premium:
        allowed = [t for t in request.tools if t.name != "book_premium_concierge"]
        request = request.override(tools=allowed)
    return handler(request)

In [6]:
# Context for the Agent
class TravelerContext(BaseModel):
    user_id: str
    home_currency: str
    membership_tier: str
    is_premium_member: bool

In [7]:
# Long-term Memory via InMemoryStore
from typing import Any
from langchain.tools import ToolRuntime
from langgraph.store.memory import InMemoryStore

travel_store = InMemoryStore()

@tool
def save_travel_style(user_id: str, style: str, runtime: ToolRuntime) -> str:
    """Save a traveler's preferred trip style (e.g. budget, luxury, adventure) for future visits."""
    runtime.store.put((user_id, "preferences"), "travel_style", {"value": style})
    return f"Noted -- I'll remember you prefer {style} travel."

@tool
def recall_travel_style(user_id: str, runtime: ToolRuntime) -> str:
    """Recall a traveler's preferred trip style, if saved before."""
    result = runtime.store.get((user_id, "preferences"), "travel_style")
    return result.value["value"] if result else "No travel style saved yet for this user."


# Test InMemoryStore
travel_store.put(("user123", "preferences"), "travel_style", {"value": "adventure"})
print(travel_store.get(("user123", "preferences"), "travel_style").value["value"])

adventure


In [8]:
# Structured Output
from pydantic import BaseModel, Field
from typing import Union, Literal
from langchain.agents.structured_output import ToolStrategy

class NewTripRequest(BaseModel):
    """A request to plan a new trip."""
    user_id: str
    destination: str
    start_date: str = Field(description="Format: YYYY-MM-DD")
    end_date: str = Field(description="Format: YYYY-MM-DD")

class ModifyTripRequest(BaseModel):
    """A request to change an existing trip."""
    user_id: str
    trip_id: int
    change_description: str

class CancelTripRequest(BaseModel):
    """A request to cancel an existing trip."""
    user_id: str
    trip_id: int

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

tripmate_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[
        get_real_weather, 
        tavily_search, 
        save_trip, 
        get_saved_trips, 
        save_travel_style, 
        recall_travel_style, 
        book_premium_concierge
    ],
    system_prompt = (
        """
        You are TripMate, a reliable travel planning assistant.

        Rules:
        1. Use web search for current or real-world information.
        2. Check real weather before making weather-dependent recommendations.
        3. Save a trip only when the user confirms the trip.
        4. Use saved trips when the user asks about their previous trips.
        5. Remember and use the user's travel style preference when relevant.
        6. Never claim a trip was saved unless the save_trip tool succeeds.
        7. Never claim weather information without calling the weather tool.
        8. Ask for missing information when required instead of guessing.
        9. Keep responses concise and useful.
        10. Avoid unnecessary tool calls. Once you have enough information to answer, respond.
        """
    ),
    middleware=[gate_premium_tools],
    checkpointer=InMemorySaver(),
    store=travel_store,
    context_schema=TravelerContext,
    # response_format=ToolStrategy(Union[NewTripRequest, ModifyTripRequest, CancelTripRequest]),
    name='advanced_tripmate_agent'
)

In [10]:
config = {"configurable": {"thread_id": "testing-session"}}

result = tripmate_agent.invoke(
    {"messages": [("user", "I'm rohan_01. What's the weather like in Bali? I prefer budget travel, please remember that.")]},
    config=config,
    context=TravelerContext(user_id="rohan_01", home_currency="INR", membership_tier="standard",is_premium_member=False),
)

print(result)

c:\Users\linto\Code\agentic-ai\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=TravelerContext(user_id='...is_premium_member=False), input_type=TravelerContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\linto\Code\agentic-ai\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=TravelerContext(user_id='...is_premium_member=False), input_type=TravelerContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content="I'm rohan_01. What's the weather like in Bali? I prefer budget travel, please remember that.", additional_kwargs={}, response_metadata={}, id='7fda6544-9a5c-47f0-9900-13d1030f103e'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 579, 'prompt_tokens': 1625, 'total_tokens': 2204, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1280}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EV1HHl5kvVqYi8WbPFrsxejMObyTd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='advanced_tripmate_agent', id='lc_run--01a1039a-0308-7462-a16f-25c98d0069ab-0', tool_calls=[{'name': 'get_real_weather', 'args': {'city': 'Bali, Indones

In [13]:
print(result["messages"][-1].content)

Hi rohan_01. Here’s the latest:

- Bali, Indonesia: 19.6°C, wind 11.2 km/h.

I've noted that you prefer budget travel. Would you like me to:
- pull a budget-friendly Bali itinerary, or
- check a 7-day forecast for your travel window, or
- suggest wallet-friendly lodging and activities? If you share your dates, I can tailor options.


In [12]:
for message in result["messages"]:
    print(f"\n--- {message.type.upper()} ---")
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:")
        for tool_call in message.tool_calls:
            print(f"  → {tool_call['name']}")
            print(f"    Args: {tool_call['args']}")


--- HUMAN ---
I'm rohan_01. What's the weather like in Bali? I prefer budget travel, please remember that.

--- AI ---

Tool calls:
  → get_real_weather
    Args: {'city': 'Bali, Indonesia'}
  → save_travel_style
    Args: {'user_id': 'rohan_01', 'style': 'budget'}

--- TOOL ---
Real current weather in Bali, Indonesia: 19.6°C, wind 11.2 km/h.

--- TOOL ---
Noted -- I'll remember you prefer budget travel.

--- AI ---
Hi rohan_01. Here’s the latest:

- Bali, Indonesia: 19.6°C, wind 11.2 km/h.

I've noted that you prefer budget travel. Would you like me to:
- pull a budget-friendly Bali itinerary, or
- check a 7-day forecast for your travel window, or
- suggest wallet-friendly lodging and activities? If you share your dates, I can tailor options.


In [16]:
config = {"configurable": {"thread_id": "new-session"}}

result = tripmate_agent.invoke(
    {"messages": [("user", "I'm rohan_02. I want to go to either Bengaluru or Jammu. I need a chill climate and also check latest news to find which one is safer. Book a trip based on safety and climate for me for December first week")]},
    config=config,
    context=TravelerContext(user_id="rohan_02", home_currency="INR", membership_tier="standard",is_premium_member=False),
)

for message in result["messages"]:
    print(f"\n--- {message.type.upper()} ---")
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:")
        for tool_call in message.tool_calls:
            print(f"  → {tool_call['name']}")
            print(f"    Args: {tool_call['args']}")


--- HUMAN ---
I'm rohan_02. I want to go to either Bengaluru or Jammu. I need a chill climate and also check latest news to find which one is safer. Book a trip for me for December first week

--- AI ---

Tool calls:
  → tavily_search
    Args: {'query': 'Bengaluru safety latest news', 'search_depth': 'advanced', 'topic': 'general', 'start_date': None, 'end_date': None, 'include_domains': None, 'exclude_domains': None, 'time_range': None, 'include_images': False}
  → tavily_search
    Args: {'query': 'Jammu safety latest news', 'search_depth': 'advanced', 'topic': 'general', 'start_date': None, 'end_date': None, 'include_domains': None, 'exclude_domains': None, 'time_range': None, 'include_images': False}
  → get_real_weather
    Args: {'city': 'Bengaluru'}
  → get_real_weather
    Args: {'city': 'Jammu'}

--- TOOL ---
{"query": "Bengaluru safety latest news", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.facebook.com/karnatakaportfolio/vi